### Chatbot And RAG Evaluation

In [35]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")    
os.environ["LANGSMITH_TRACING"]="true"

In [36]:
# Create the datapoints - these all will be directly created in LangSmith
from langsmith import Client
ls_client = Client()

# define the dataset
dataset_name = "Simple Chatbot Evaluation"

# FIXED: Safely handle dataset creation if it already exists to prevent 409 conflict
try:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists. Skipping creation.")
except Exception:
    dataset = ls_client.create_dataset(dataset_name)
    ls_client.create_examples(
        dataset_id=dataset.id,
        examples=[
            {
                "inputs": {"question": "What is LangChain?"},  
                "outputs": {"answer": "A framework for building LLM applications"},
            },
            {
                "inputs": {"question": "What is LangSmith?"},
                "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
            },
            {
                "inputs": {"question": "What is OpenAI?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            },
            {
                "inputs": {"question": "What is Google?"},
                "outputs": {"answer": "A technology company known for search"},
            },
            {
                "inputs": {"question": "What is Mistral?"},
                "outputs": {"answer": "A company that creates Large Language Models"},
            }
        ]
    )
    print(f"Dataset '{dataset_name}' created successfully with examples.")

Dataset 'Simple Chatbot Evaluation' already exists. Skipping creation.


### Define Metric (LLM as Judge)

In [37]:
from google import genai
from langsmith import wrappers

gemini_base_client = genai.Client() # reads GOOGLE_API_KEY from environment variables
gemini_client = wrappers.wrap_gemini(gemini_base_client)

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:  
    """
      # Grade: equals that specific value automatically
      response=gemini_client.chat.completions.create(
            model="gemini-2.5-flash",  # this is making the deicison
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content # read last msg

      return response == "CORRECT" # if response is correct, return true

In [38]:
## Concisions (2nd metric)- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run Evaluation

In [ ]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
# app chatbot application
def my_app(question: str, model: str = "gemini-2.5-flash", instructions: str = default_instructions) -> str:
    return gemini_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [43]:
# Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
# Run evaluation
# from langsmith import evaluate
experiment_results = ls_client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="gemini-2.5-flash-chatbot"
)

Error running target function: 'Chats' object has no attribute 'completions'
Traceback (most recent call last):
  File "d:\ai-learning-roadmap\complete_agentic_ai\llmevaluation\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 2062, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ai-learning-roadmap\complete_agentic_ai\llmevaluation\.venv\Lib\site-packages\langsmith\run_helpers.py", line 849, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_22840\2991587408.py", line 3, in ls_target
    return {"response": my_app(inputs["question"])}
                        ~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_22840\3111188798.py", line 4, in my_app
    return gemini_client.chats.completions.create(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'Chats' object 

View the evaluation results for experiment: 'gemini-chatbot-259bf835' at:
https://smith.langchain.com/o/e0e0cb97-527f-42d9-9b12-9fc2709c896e/datasets/d4561054-2120-4581-8d8f-08ad1200f70b/compare?selectedSessions=3427ae7f-3df8-4b28-905d-3e1131e29d01




In [46]:
### now testing with another model
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"],model="gemini-3.5-flash")}

In [47]:
## Run our evaluation
experiment_results=ls_client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="gemini-3.5-flash-chatbot"
)

Error running target function: 'Chats' object has no attribute 'completions'
Traceback (most recent call last):
  File "d:\ai-learning-roadmap\complete_agentic_ai\llmevaluation\.venv\Lib\site-packages\langsmith\evaluation\_runner.py", line 2062, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\ai-learning-roadmap\complete_agentic_ai\llmevaluation\.venv\Lib\site-packages\langsmith\run_helpers.py", line 849, in wrapper
    function_result = run_container["context"].run(
        func, *args, **kwargs
    )
  File "C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_22840\3728453039.py", line 3, in ls_target
    return {"response": my_app(inputs["question"],model="gemini-3.5-flash")}
                        ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_22840\3111188798.py", line 4, in my_app
    return gemini_client.chats.completions.create(
           ^^^^^^^^^^^^^

View the evaluation results for experiment: 'gemini-3.5-flash-chatbot-d41b830f' at:
https://smith.langchain.com/o/e0e0cb97-527f-42d9-9b12-9fc2709c896e/datasets/d4561054-2120-4581-8d8f-08ad1200f70b/compare?selectedSessions=0a4bbd6b-6631-48e5-a0ae-f7c7c1b0c984


